# Solutions - Set Operations (Easy 15)


In [ ]:
from pyspark.sql import functions as F, types as T

customers = spark.table("samples.bakehouse.sales_customers")
tpch_customer = spark.table("samples.tpch.customer")
users = spark.table("samples.wanderbricks.users")


## Problem 1 - Union Countries

Combine distinct countries from bakehouse and wanderbricks with a source label.


In [ ]:
bk = customers.select("country").distinct().withColumn("source", F.lit("bakehouse"))
wb = users.select("country").distinct().withColumn("source", F.lit("wanderbricks"))
result_1 = bk.unionByName(wb)
result_1.display()


## Problem 2 - Intersect Countries

Find countries that appear in both bakehouse and wanderbricks.


In [ ]:
bk = customers.select("country").distinct()
wb = users.select("country").distinct()
result_2 = bk.intersect(wb)
result_2.display()


## Problem 3 - Subtract Countries

Find countries in bakehouse that are not in wanderbricks.


In [ ]:
bk = customers.select("country").distinct()
wb = users.select("country").distinct()
result_3 = bk.subtract(wb)
result_3.display()


## Problem 4 - Dedup Summary

Compare row counts before and after deduplicating on name and country.


In [ ]:
before = customers.count()
after = customers.dropDuplicates(["first_name", "last_name", "country"]).count()
result_4 = spark.createDataFrame(
    [("before_dedup", before), ("after_dedup", after)],
    ["step", "row_count"]
)
result_4.display()


## Problem 5 - Union All Sources Grouped

Combine bakehouse customers and wanderbricks users into a unified dataset and count by source.


In [ ]:
bk = customers.select(
    F.col("customerID").alias("user_id"),
    F.col("first_name").alias("name"),
    "country",
    F.lit("bakehouse").alias("source")
)
wb = users.select(
    "user_id", "name", "country",
    F.lit("wanderbricks").alias("source")
)
result_5 = bk.unionByName(wb).groupBy("source").agg(
    F.count("*").alias("user_count")
)
result_5.display()
